# Practical 6 — Named Entity Recognition (NER)

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To identify named entities (people, organizations, dates, money, etc.) in review text using NLTK and spaCy, and to test what happens to NER accuracy when it's run on Practical 1's cleaned/lowercased tokens instead of raw text.

## Theory

**Named Entity Recognition (NER)** locates and classifies spans of text that refer to real-world entities — people (PERSON), organizations (ORG), locations (GPE), dates (DATE), monetary values (MONEY), and so on.

Two approaches compared here:
- **NLTK's `ne_chunk`** — statistical, built on top of POS tagging (similar mechanism to Practical 5's chunking, but trained specifically to recognize entity spans rather than generic noun phrases).
- **spaCy's NER pipeline** — a trained neural model, generally more accurate on real-world text.

Here's the part worth thinking about before running anything: every practical so far has reused Practical 1's cleaned tokens (lowercased, punctuation and numbers stripped) as the starting point. NER is different from everything covered so far in one important way — **capitalization is one of the strongest signals** a NER system uses to recognize a proper noun ("Robert De Niro" vs "robert de niro"), and punctuation/number preservation matters for recognizing dates and monetary values. This practical exists specifically to test whether feeding NER the same cleaned tokens used in every previous practical was ever actually a good idea.

## Algorithm

1. Pick a review containing a real named entity (review 4, which mentions "Robert De Niro").
2. Run both NLTK and spaCy NER on the **raw, original** review text.
3. Run both on the **same review after being passed through Practical 1's `clean_text()`** (lowercased, punctuation/numbers stripped).
4. Compare all four results directly.
5. Run spaCy NER (the more robust of the two) across all 15 **raw** reviews and tally what entity types actually show up in this dataset.

In [1]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

for pkg in ["punkt", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    try:
        nltk.download(pkg)
    except Exception as e:
        print(f"Skipped {pkg}: {e}")

import preprocessing
import ner

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
print(f"Loaded {len(df)} reviews")


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package words to
[nltk_data]     /Users/hayasachin/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


Loaded 15 reviews


### Step 1 — NER on raw text vs cleaned text, same review

In [2]:
raw_review = df[df["id"] == 4]["review"].iloc[0]
cleaned_review = preprocessing.clean_text(raw_review)

print(f"RAW:     {raw_review}")
print(f"CLEANED: {cleaned_review}")
print()

result = ner.compare_raw_vs_cleaned(raw_review, cleaned_review)

print("NLTK on RAW:    ", result["nltk_on_raw"])
print("NLTK on CLEANED:", result["nltk_on_cleaned"])
print()
print("spaCy on RAW:    ", result["spacy_on_raw"])
print("spaCy on CLEANED:", result["spacy_on_cleaned"])


RAW:     I can't believe how good the acting was!! Robert De Niro really outdid himself this time.
CLEANED: i cant believe how good the acting was robert de niro really outdid himself this time

NLTK on RAW:     [('Robert De Niro', 'PERSON')]
NLTK on CLEANED: []

spaCy on RAW:     [('Robert De Niro', 'PERSON')]
spaCy on CLEANED: [('robert de niro', 'PERSON')]


**Check specifically: did either tagger still correctly identify "Robert De Niro" (or any part of it) as a PERSON once the text was lowercased and stripped of punctuation? Note exactly what each of the four results actually returned.**

### Step 2 — Entity types found across the raw dataset (spaCy)

In [3]:
from collections import Counter

entity_counter = Counter()
entities_by_review = {}

for _, row in df.iterrows():
    ents = ner.spacy_ner(row["review"])
    if ents:
        entities_by_review[row["id"]] = ents
    entity_counter.update(label for text, label in ents)

print("Entity type counts across all 15 raw reviews:")
for label, count in entity_counter.most_common():
    print(f"  {label}: {count}")

print()
print("Entities found, by review:")
for review_id, ents in entities_by_review.items():
    print(f"  Review {review_id}: {ents}")


Entity type counts across all 15 raw reviews:
  DATE: 5
  ORG: 4
  CARDINAL: 4
  MONEY: 3
  PERSON: 1
  ORDINAL: 1
  TIME: 1

Entities found, by review:
  Review 1: [('ABSOLUTELY', 'ORG')]
  Review 2: [('2026', 'DATE'), ('12', 'MONEY')]
  Review 3: [('7/10', 'CARDINAL')]
  Review 4: [('Robert De Niro', 'PERSON')]
  Review 6: [('3rd', 'ORDINAL')]
  Review 7: [('Two hours', 'TIME'), ('15', 'CARDINAL'), ('2/10', 'CARDINAL')]
  Review 8: [('WOW', 'ORG'), ('years', 'DATE')]
  Review 10: [('these days', 'DATE'), ('18.50', 'MONEY'), ('every penny', 'MONEY')]
  Review 12: [('the decade, period', 'DATE')]
  Review 13: [('BAD', 'ORG')]
  Review 14: [('5', 'CARDINAL')]
  Review 15: [('CGI', 'ORG'), ('2010', 'DATE')]


## Observations & Conclusion
Across the 15 raw reviews, DATE (5), ORG (4), CARDINAL (4), and MONEY (3) were the most common entity types found, with only one genuine PERSON entity ("Robert De Niro" in review 4). Comparing NER on raw vs. cleaned text for that review showed a clear asymmetry: NLTK failed to detect the entity at all once the text was lowercased and stripped of punctuation, while spaCy still correctly identified "robert de niro" as PERSON despite the lowercasing — showing spaCy is meaningfully more robust to the kind of aggressive cleaning used in Practical 1. A separate and distinct failure mode showed up in the false positives: "ABSOLUTELY," "WOW," "BAD," and "CGI" were all misclassified as ORG, and all four share the same visual pattern as real acronyms and organization names (NASA, IBM, BBC) — fully capitalized tokens. This is the opposite problem from the lowercasing case: rather than losing a needed signal, the model over-relies on a capitalization pattern that happens to coincide with informal emphasis (all-caps for emphasis) rather than an actual entity. Together these show that NER is highly sensitive to capitalization in two different, opposite ways — losing it causes missed entities, and unusual-but-plausible-looking capitalization causes false ones — which strongly suggests NER should run on raw text before any lowercasing or punctuation stripping, and likely needs a real pipeline to branch into separately-processed versions of the text for different tasks rather than reusing one cleaned version for everything.

---
## Viva Prep — Practice Questions

1. **What is Named Entity Recognition, and name three common entity types.**
   NER locates and classifies spans of text referring to real-world entities. Common types: PERSON, ORG (organization), GPE (geopolitical entity/location), DATE, MONEY.

2. **Why does capitalization matter so much for NER, specifically for English text?**
   In English, proper nouns are conventionally capitalized, and traditional/statistical NER systems use capitalization as a strong feature to distinguish a named entity ("Paris" the city) from a common word ("paris" wouldn't naturally occur, but the pattern generalizes to names like "Robert" vs "robert" not standing out from ordinary lowercase text).

3. **Why would running NER on Practical 1's cleaned tokens be a bad idea?**
   `clean_text()` lowercases all text and strips punctuation/numbers — removing exactly the signals (capitalization for names, punctuation and digit patterns for dates/money) that NER systems rely on most.

4. **What's the difference between NLTK's `ne_chunk` and spaCy's NER pipeline, mechanically?**
   NLTK's approach layers a statistical chunker on top of POS tags (similar mechanism to the noun-phrase chunking in Practical 5, but trained for entity spans); spaCy uses a trained neural model that considers more context directly, generally with higher accuracy on real-world text.

5. **If you had to redesign Practicals 1-6 as a single pipeline, where would NER need to run relative to the cleaning step?**
   NER would need to run on the raw or minimally-processed text, before (or independent of) the aggressive lowercasing/punctuation-stripping pipeline used for the other tasks — meaning a real system likely needs to branch into two differently-processed versions of the text rather than using one cleaned version for everything.
